<a href="https://colab.research.google.com/github/aksonajswl/Word-Predictor/blob/main/Word_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Full Pipeline: From .txt to Word Prediction**

In [36]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
import re
import random
import numpy as np

#Setting seed


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)  # ✅ Call it at the start




**Load and Preprocess the Text**

In [37]:
# Load text file
with open('/content/Dracula.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Preprocess: lowercase, remove punctuation, tokenize
tokens = re.findall(r'\b\w+\b', text.lower())  # word-level tokenization


print(tokens[:10])
print(len(tokens))

['how', 'these', 'papers', 'have', 'been', 'placed', 'in', 'sequence', 'will', 'be']
163498


**Build Vocabulary**

In [38]:
word_counts = Counter(tokens)
vocab = sorted(word_counts, key=word_counts.get, reverse=True)
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(vocab)

print(f"Vocab -  {vocab[:10]}\n")
print(f"Vocabulary Size = {vocab_size}\n")

print(list(word2idx.items())[:10])

print("\n")

print(list(idx2word.items())[:10])


Vocab -  ['the', 'and', 'i', 'to', 'of', 'a', 'he', 'in', 'that', 'it']

Vocabulary Size = 9443

[('the', 0), ('and', 1), ('i', 2), ('to', 3), ('of', 4), ('a', 5), ('he', 6), ('in', 7), ('that', 8), ('it', 9)]


[(0, 'the'), (1, 'and'), (2, 'i'), (3, 'to'), (4, 'of'), (5, 'a'), (6, 'he'), (7, 'in'), (8, 'that'), (9, 'it')]


**Create Input Sequences and Targets**

In [53]:
sequence_length = 5  # You can change this

data = []
for i in range(len(tokens) - sequence_length):
    input_seq = tokens[i:i+sequence_length]
    target_word = tokens[i+sequence_length]
    data.append((
        [word2idx[word] for word in input_seq],
        word2idx[target_word]
    ))


set_seed(42)
# Shuffle for training
random.shuffle(data)

print(data[:10])
print(len(data))


[([74, 75, 270, 9, 95], 185), ([27, 248, 79, 168, 178], 513), ([4817, 866, 41, 1314, 38], 16), ([1360, 754, 35, 41, 281], 24), ([26, 19, 763, 1, 1567], 125), ([956, 957, 70, 71, 38], 286), ([1109, 82, 0, 174, 6], 78), ([448, 94, 3, 255, 3], 161), ([314, 4488, 635, 101, 2121], 1), ([1832, 41, 188, 213, 3], 3331)]
163493


**Prepare Tensors**

In [54]:
X = torch.tensor([x for x, _ in data])       # shape: (num_samples, seq_len)
y = torch.tensor([y for _, y in data])       # shape: (num_samples,)


**Define the Model**

In [57]:
class WordPredictor(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embed(x)               # (batch, seq_len, embed_dim)
        out, _ = self.rnn(x)            # out: (batch, seq_len, hidden)
        out = self.fc(out[:, -1, :])    # use last output
        return out


** Training Loop**

In [58]:
# Hyperparameters
embedding_dim = 50
hidden_size = 128
num_epochs = 5
batch_size = 64

set_seed(42)             # Set seed first

model = WordPredictor(vocab_size, embedding_dim, hidden_size)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    # Mini-batch training
    for i in range(0, len(X), batch_size):
        x_batch = X[i:i+batch_size]
        y_batch = y[i:i+batch_size]

        optimizer.zero_grad()
        output = model(x_batch)
        loss = loss_fn(output, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / (len(X) / batch_size)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")


Epoch 1, Loss: 6.0319
Epoch 2, Loss: 5.3459
Epoch 3, Loss: 5.0217
Epoch 4, Loss: 4.7670
Epoch 5, Loss: 4.5502


# *Predict the Next Word*

In [65]:
def predict_next(model, text_snippet):
    model.eval()
    words = text_snippet.lower().split()
    words = words[-sequence_length:]  # take last N words
    input_idx = [word2idx.get(w, 0) for w in words]

    if len(input_idx) < sequence_length:
        input_idx = [0] * (sequence_length - len(input_idx)) + input_idx

    input_tensor = torch.tensor([input_idx])
    with torch.no_grad():
        output = model(input_tensor)
        predicted_idx = torch.argmax(output, dim=1).item()
        return idx2word[predicted_idx]

# Example:
print(predict_next(model, "he went to the"))
print(predict_next(model, "the story was so"))
print(predict_next(model, "I found that my"))
print(predict_next(model, "Just before I was"))
print(predict_next(model, "green and brown where"))

window
bright
heart
not
the
